# 🕵️ Dia 88 — O grafo final do caso

**Investigador Cyber 90 Dias** · Fase 7 · Rank Jutsu `S`

> Baseado no *Day 88* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-7-capstone-career/day-88.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).
> **Material sintético.** O *case packet* (Orrin Valley / Castellan) e a Operação Isca são ficção; domínios `.example`, IPs de documentação.

---

### 🎬 Onde paramos

Rafa fotografou a parede de post-its e jogou tudo fora: *"A Marta e o delegado vão ver uma imagem e uma legenda. Em dois minutos, alguém de fora precisa dizer como o dinheiro saiu e qual parte da história é a mais fraca."*

Neste notebook você vai:

1. **Montar o grafo do capstone** a partir das linhas resolvidas do original, e acrescentar as suas.
2. **Rodar o `check_graph.py` do original** e ver o que ele pega quando você erra de propósito.
3. **Calcular grau, clusters e pontes** com Python puro.
4. **Achar o caminho confirmado** da imitação até o pagamento (busca em largura).
5. **Gerar o grafo em Mermaid**, com cor por confiança e legenda.
6. **Fazer tudo no grafo final da Operação Isca.**

🐍 Só biblioteca padrão do Python. Funciona offline: se a rede falhar, o notebook usa a cópia embutida dos arquivos. Rode as células de cima para baixo (Shift+Enter).

## 1. O ponto de partida: as linhas resolvidas do original

As cinco arestas do Passo 2 do original, mais algumas que saem **diretamente** de evidências que você já leu nos Dias 84 a 87 (cada uma com a citação). As arestas que o original deixa para você classificar (as pontes, o vazamento, a infraestrutura achada por pivô) **não vêm prontas**: você as acrescenta no formulário da seção 2.

In [ ]:
import csv, io
from pathlib import Path

NOS = """Id,Label,type,cluster
d_lookalike,castellan-hardwood[.]example,domain,castellan
d_phish,cstl-docshare[.]example,domain,castellan
d_real,castellanhardwood[.]example,domain,victim
ip_023,198.51.100[.]23,ip,bridge
ip_047,203.0.113[.]47,ip,castellan
msg_M1,M1 real invoice 3 Mar,message,victim
msg_M2,M2 phishing email 10 Mar,message,castellan
msg_M3,M3 fake bank letter 11 Mar,message,castellan
msg_M7,M7 remittance advice 13 Mar,message,victim
sess_7f3a61,session s-7f3a61,session,castellan
sess_7f9b04,session s-7f9b04,session,castellan
acct_jpike,jpike mailbox,account,victim
acct_mbeaulieu,mbeaulieu mailbox,account,victim
rule_dot,inbox rule '.',mailbox_object,castellan
fin_8830,payee account ending 8830,financial,castellan
pay_000731,payment 000731,financial,victim
email_ar,ar@castellan-hardwood[.]example,email_address,castellan
x_spray,203.0.113[.]200 (password spray),ip,excluded
x_cohost,1412 co-hosted domains on 203.0.113[.]88,domain,excluded
"""

ARESTAS = """Source,Target,Type,Label,evidence,confidence,day
d_lookalike,ip_047,Directed,A record since 2026-02-25,P6 L3,confirmed,84
ip_023,msg_M2,Directed,authenticated submitter of,P2 lowest Received header,confirmed,84
msg_M2,d_phish,Directed,links to (href),P2 HTML part; P9 10 Mar 14:19:07 UTC,confirmed,84
ip_023,sess_7f3a61,Directed,sign-in source,P8 2026-03-10T14:31:12Z,confirmed,86
sess_7f3a61,acct_jpike,Directed,signed in to,P8 2026-03-10T14:31:12Z,confirmed,86
msg_M2,d_lookalike,Directed,sent from domain,P2 From; P7 2026-03-10T14:02:37Z,confirmed,84
msg_M2,msg_M1,Directed,In-Reply-To quotes,P2 In-Reply-To; P3 M1 Message-ID,confirmed,84
msg_M1,d_real,Directed,sent from domain (SPF DKIM DMARC pass),P3 M1; P7 2026-03-03T14:47:04Z,confirmed,84
sess_7f3a61,rule_dot,Directed,created rule,P8 2026-03-10T14:36:50Z,confirmed,86
ip_023,sess_7f9b04,Directed,sign-in source,P8 2026-03-11T13:47:20Z,confirmed,86
sess_7f9b04,acct_jpike,Directed,signed in to,P8 2026-03-11T13:47:20Z,confirmed,86
sess_7f9b04,msg_M3,Directed,sent,P8 2026-03-11T13:52:44Z; P7 13:52:45Z,confirmed,86
msg_M3,acct_mbeaulieu,Directed,delivered to,P7 2026-03-11T13:52:45Z; P3 M3,confirmed,86
msg_M3,fin_8830,Directed,names payee account,P3 M3 letter text,confirmed,87
acct_mbeaulieu,email_ar,Directed,vendor record changed to,P3 M4; P1 Part C (Maren statement),likely,87
pay_000731,fin_8830,Directed,remitted to,P3 M7,confirmed,87
msg_M7,email_ar,Directed,remittance advice sent to,P7 2026-03-13T19:02:09Z; P3 M7,confirmed,87
msg_M7,pay_000731,Directed,announces,P3 M7,confirmed,87
d_lookalike,email_ar,Directed,address on domain,P2 Authenticated sender; P5 B1,confirmed,84
x_spray,acct_jpike,Directed,failed spray; no link shown; rejected,P8 2026-03-10T03:11:40Z,excluded,86
x_cohost,d_phish,Directed,co-hosted on shared server; rejected,P6 L5,excluded,84
"""

def ler(texto):
    return list(csv.DictReader(io.StringIO(texto)))

nos, arestas = ler(NOS), ler(ARESTAS)
print(f"{len(nos)} nós, {len(arestas)} arestas no ponto de partida")

## 2. Acrescente as suas arestas

Use o formulário para acrescentar as arestas que o original deixa para você (as roxas com `?` do diagrama): a ligação com a infraestrutura Marrowline, o hash do JS, as hipóteses do vazamento. Uma aresta por rodada; rode a célula várias vezes. **Toda aresta precisa de evidência**, e o nível de confiança é decisão sua, a partir das suas notas.

In [ ]:
#@title Nova aresta (rode uma vez por aresta) { display-mode: "form" }
novo_no_id = "d_marrow"  #@param {type:"string"}
novo_no_rotulo = "marrowline-docs[.]example"  #@param {type:"string"}
novo_no_tipo = "domain"  #@param ["domain","ip","email_address","phone","artifact","message","account","session","mailbox_object","financial","organization","hypothesis"]
novo_no_cluster = "marrowline"  #@param {type:"string"}
origem = "ip_023"  #@param {type:"string"}
destino = "d_marrow"  #@param {type:"string"}
rotulo = "mail.marrowline-docs A record until 2026-03-09"  #@param {type:"string"}
evidencia = "P6 L3"  #@param {type:"string"}
confianca = "likely"  #@param ["confirmed","likely","possible","excluded"]
dia = "84"  #@param {type:"string"}

if novo_no_id and novo_no_id not in {n["Id"] for n in nos}:          # cria o nó se ainda não existe
    nos.append({"Id": novo_no_id, "Label": novo_no_rotulo, "type": novo_no_tipo, "cluster": novo_no_cluster})
nova = {"Source": origem, "Target": destino, "Type": "Directed", "Label": rotulo,
        "evidence": evidencia, "confidence": confianca, "day": dia}
if nova not in arestas:                                                # não duplica se rodar duas vezes
    arestas.append(nova)
print(f"{len(nos)} nós, {len(arestas)} arestas. Última: {origem} -> {destino} [{confianca}] ({evidencia})")

## 3. O `check_graph.py` do original

Gravamos os CSVs e rodamos **o script do original, sem mudança**. Depois, de propósito, criamos uma aresta com um `Id` digitado errado e sem evidência, para ver o verificador funcionando.

In [ ]:
CHECK_GRAPH = """import csv

LEVELS = {"confirmed", "likely", "possible", "excluded"}
with open("nodes.csv", newline="") as f:
    nodes = list(csv.DictReader(f))
ids = [n["Id"] for n in nodes]
problems = 0
for dup in sorted({i for i in ids if ids.count(i) > 1}):
    print(f"nodes.csv: duplicate Id {dup}"); problems += 1
with open("edges.csv", newline="") as f:
    for line, e in enumerate(csv.DictReader(f), start=2):
        for end in ("Source", "Target"):
            if e[end] not in ids:
                print(f"edges.csv line {line}: {end} '{e[end]}' is not a node Id"); problems += 1
        if not e["evidence"].strip():
            print(f"edges.csv line {line}: no evidence citation"); problems += 1
        if e["confidence"] not in LEVELS:
            print(f"edges.csv line {line}: confidence '{e['confidence']}' not in {sorted(LEVELS)}"); problems += 1
used = set()
with open("edges.csv", newline="") as f:
    for e in csv.DictReader(f):
        used.update((e["Source"], e["Target"]))
for orphan in sorted(set(ids) - used):
    print(f"nodes.csv: {orphan} has no edges; link it or drop it")
print(f"{len(ids)} nodes checked, {problems} problems")
"""

def gravar(pasta, nos, arestas):
    pasta = Path(pasta); pasta.mkdir(parents=True, exist_ok=True)
    for nome, linhas in (("nodes.csv", nos), ("edges.csv", arestas)):
        with open(pasta / nome, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(linhas[0].keys()))
            w.writeheader(); w.writerows(linhas)
    (pasta / "check_graph.py").write_text(CHECK_GRAPH, encoding="utf-8")

import os, contextlib
def rodar_check(pasta):
    """Roda o check_graph.py de dentro da pasta, como o original manda."""
    antes = os.getcwd()
    try:
        os.chdir(pasta)
        exec(compile(Path("check_graph.py").read_text(encoding="utf-8"), "check_graph.py", "exec"), {})
    finally:
        os.chdir(antes)

gravar("notes/graph", nos, arestas)
print("== Seu grafo:"); rodar_check("notes/graph")

print("\n== Agora com dois erros de propósito:")
errada = dict(arestas[0], Target="ip_O47", evidence="")              # letra O no lugar do zero
gravar("teste-erro", nos, arestas + [errada])
rodar_check("teste-erro")

## 4. Leitura do grafo: grau, clusters e pontes (sem Gephi)

In [ ]:
from collections import Counter, defaultdict, deque

def ler_grafo(nos, arestas, incluir_excluidos=False):
    cluster = {n["Id"]: n["cluster"] for n in nos}
    rotulo = {n["Id"]: n["Label"] for n in nos}
    validas = [e for e in arestas if incluir_excluidos or e["confidence"] != "excluded"]
    grau = Counter()
    for e in validas:
        grau[e["Source"]] += 1; grau[e["Target"]] += 1
    print("Hubs (maior grau):")
    for nid, g in grau.most_common(5):
        print(f"   {g:2}  {rotulo[nid]:42} [{cluster[nid]}]")
    print("\nNós por cluster:", dict(Counter(cluster.values())))
    print("\nArestas que cruzam clusters (candidatas a ponte):")
    for e in validas:
        a, b = cluster[e["Source"]], cluster[e["Target"]]
        if a != b:
            print(f"   {rotulo[e['Source']]} [{a}] -> {rotulo[e['Target']]} [{b}]  · {e['Label']} · {e['confidence']} · {e['evidence']}")
    excl = [n for n in nos if n["cluster"] == "excluded"]
    print(f"\nExcluídos visíveis: {[n['Label'] for n in excl]}")

ler_grafo(nos, arestas)

> 🧠 Repare se o maior hub é um nó como `jpike mailbox`, em que "tudo encosta" por construção. O original pede para separar hubs que importam (um IP do atacante) de hubs que são artefato do desenho.

## 5. O caminho confirmado: da imitação ao pagamento

Checkpoint 6 do original: *um caminho de arestas confirmadas vai do registro do domínio imitação até o pagamento 000731*. A busca em largura abaixo anda pelas arestas **nos dois sentidos** (um caminho de prova pode passar por uma aresta "contra a mão"), mas só pelas confirmadas.

In [ ]:
def caminho(nos, arestas, de, ate, niveis=("confirmed",)):
    viz = defaultdict(list)
    for e in arestas:
        if e["confidence"] in niveis:
            viz[e["Source"]].append((e["Target"], e))
            viz[e["Target"]].append((e["Source"], e))      # ida e volta
    fila, visto = deque([(de, [])]), {de}
    while fila:
        atual, trilha = fila.popleft()
        if atual == ate:
            return trilha
        for prox, e in viz[atual]:
            if prox not in visto:
                visto.add(prox); fila.append((prox, trilha + [e]))
    return None

def caminho_por(nos, arestas, pontos, niveis=("confirmed",)):
    """Caminho que passa, em ordem, pelos pontos dados. O mais curto puro pega atalhos
    (ex.: imitação -> endereço ar@ -> pagamento) e pula a história; os pontos obrigam a contá-la."""
    total = []
    for a, b in zip(pontos, pontos[1:]):
        trecho = caminho(nos, arestas, a, b, niveis)
        if trecho is None:
            return None
        total += trecho
    return total

rot = {n["Id"]: n["Label"] for n in nos}
print("Caminho mais curto (repare no atalho):")
for e in caminho(nos, arestas, "d_lookalike", "pay_000731") or []:
    print(f"   {rot[e['Source']]} --[{e['Label']}]--> {rot[e['Target']]}")

PONTOS = ["d_lookalike", "msg_M2", "acct_jpike", "msg_M3", "pay_000731"]   # a história: phishing, tomada, carta, pagamento
t = caminho_por(nos, arestas, PONTOS)
print("\nCaminho só de arestas CONFIRMADAS passando por", PONTOS, ":", "nenhum" if t is None else "")
for e in t or []:
    print(f"   {rot[e['Source']]} --[{e['Label']} · {e['evidence']}]--> {rot[e['Target']]}")

## 6. O grafo em Mermaid, com legenda

O texto abaixo vai num bloco ` ```mermaid ` do GitHub (ou no [mermaid.live](https://mermaid.live/)). Cores por confiança, como no segundo diagrama do original; excluídos em cinza.

In [ ]:
ESTILO = {"confirmed": "stroke:#1f77b4,stroke-width:4px", "likely": "stroke:#2ca02c,stroke-width:2px",
          "possible": "stroke:#ff7f0e,stroke-width:2px,stroke-dasharray:4 4", "excluded": "stroke:#bbbbbb,stroke-width:1px"}
SETA = {"confirmed": "==>", "likely": "-->", "possible": "-.->", "excluded": "-->"}

def mermaid(nos, arestas):
    def seguro(t):
        return t.replace('"', "'").replace(":", "#58;")         # aspas e dois-pontos quebram o Mermaid
    linhas = ["flowchart LR"]
    for cl in sorted({n["cluster"] for n in nos}):
        linhas.append(f'    subgraph {cl}["{cl}"]')
        linhas += [f'        {n["Id"]}["{seguro(n["Label"])}"]' for n in nos if n["cluster"] == cl]
        linhas.append("    end")
    for e in arestas:
        linhas.append(f'    {e["Source"]} {SETA[e["confidence"]]}|"{seguro(e["Label"])} · {seguro(e["evidence"])}"| {e["Target"]}')
    for i, e in enumerate(arestas):
        linhas.append(f"    linkStyle {i} {ESTILO[e['confidence']]}")
    return "\n".join(linhas)

LEGENDA = """Legenda: azul grossa = confirmado (2+ fontes independentes) · verde = provável (1 fonte ou raciocínio) ·
laranja tracejada = possível (sem fonte direta) · cinza = excluído (motivo no rótulo). MATERIAL SINTÉTICO."""
texto_mermaid = mermaid(nos, arestas)
print(texto_mermaid[:1500], "\n...\n")
print(LEGENDA)
Path("notes/graph/case-graph.mmd").write_text(texto_mermaid + "\n", encoding="utf-8")

## 7. ➕ O grafo final da Operação Isca

In [ ]:
# Dados sintéticos da Operação Isca deste dia (plano B embutido).
ISCA_EMBUTIDO = {
"nos.csv": "Id,Label,type,cluster\norg_sa,Cooperativa Serra Azul (alvo),organization,vitima\nd_real,serraazul[.]example (site real),domain,vitima\nd_pix,serraazul-pix[.]example,domain,isca\nd_seg,serraazul-seguranca[.]example,domain,isca\nd_coop,coopserra-azul[.]example,domain,isca\nd_vale,coopvale-pix[.]example,domain,outras-coops\norg_vale,Coopvale (alvo em outra campanha),organization,outras-coops\nip_077,203.0.113[.]77,ip,isca\nip_150,203.0.113[.]150,ip,ponte\nart_ik,ik.js (mesmo SHA-256),artifact,ponte\ncert_1,certificado TLS 2026-10-04,artifact,isca\nsms_1,SMS 'Pix bloqueado' (4 remetentes),message,isca\ntel_1,+55 00 00000-0001 a -0004,phone,isca\nwa_1,WhatsApp 'Serra Azul Atendimento',account,isca\nv1,cooperado V1,account,vitima\nv2,cooperado V2,account,vitima\nv3,cooperado V3,account,vitima\nl01,PIXFICT-L01 (laranja L-01),financial,laranjas\nl02,PIXFICT-L02 (laranja L-02),financial,laranjas\nl03,PIXFICT-L03 (laranja L-03),financial,laranjas\nh_lia,H4 Lia usada para validar celulares,hypothesis,pergunta-aberta\nh_forn,H1 base vazou de fornecedor,hypothesis,pergunta-aberta\nx_lnk,lnk[.]example (encurtador publico),domain,excluded\nx_pous,pousada-ipe[.]example (vizinho de hospedagem),domain,excluded\nx_fav,favicon igual ao do site real,artifact,excluded\n",
"arestas.csv": "Source,Target,Type,Label,evidence,confidence,day\nsms_1,d_pix,Directed,link no texto,I2 linhas 07:12 e 07:58,confirmed,83\ntel_1,sms_1,Directed,remetentes,I2 todas as linhas,confirmed,83\nd_pix,ip_077,Directed,registro A,V2 DNS passivo,confirmed,84\nd_seg,ip_077,Directed,registro A,V2 DNS passivo,confirmed,84\nd_pix,cert_1,Directed,nome no certificado,V1 CT 2026-10-04,confirmed,84\nd_seg,cert_1,Directed,nome no certificado,V1 CT 2026-10-04,confirmed,84\nd_pix,art_ik,Directed,carrega o script,I3 RECURSO; V3,confirmed,84\nd_coop,art_ik,Directed,carrega o script,V3 varredura de URL,likely,84\nd_vale,art_ik,Directed,carrega o script,V4 varredura publica de terceiro,likely,84\nd_vale,org_vale,Directed,imita a marca,V4 texto da pagina,likely,84\nd_pix,org_sa,Directed,imita a marca,I3 TITULO,confirmed,83\nd_real,org_sa,Directed,site oficial,I1 contrato,confirmed,83\nwa_1,org_sa,Directed,imita a marca,S3 do Dia 87,likely,87\nv1,d_pix,Directed,digitou a senha,OUV-0004; IB 11:03:40Z dispositivo novo,confirmed,86\nv2,d_pix,Directed,digitou a senha,OUV-0011; IB 11:12:30Z dispositivo novo,confirmed,86\nip_150,v1,Directed,login com credencial roubada,IB 2026-10-05T11:03:40Z,confirmed,86\nip_150,v2,Directed,login com credencial roubada,IB 2026-10-05T11:12:30Z,confirmed,86\nip_150,v3,Directed,login com credencial roubada,IB 2026-10-05T11:41:19Z,confirmed,86\nv1,l01,Directed,Pix nao reconhecido,IB 11:05:12Z; contestacao PIXFICT-E2E-0001,confirmed,86\nv2,l01,Directed,Pix nao reconhecido,IB 11:14:05Z; contestacao PIXFICT-E2E-0002,confirmed,86\nv2,l02,Directed,Pix nao reconhecido,IB 11:15:44Z; contestacao PIXFICT-E2E-0003,confirmed,86\nv3,l03,Directed,Pix nao reconhecido,IB 11:43:02Z; contestacao PIXFICT-E2E-0004,confirmed,86\nv3,d_pix,Directed,digitou a senha,IB 11:41:19Z dispositivo novo (sem relato de clique na ouvidoria),likely,86\nh_lia,sms_1,Directed,explicaria quem recebeu,ACH E3 E6 do Dia 85,likely,85\nh_forn,sms_1,Directed,explicaria quem recebeu,ACH E2 E6 do Dia 85,possible,85\nx_lnk,sms_1,Directed,encurtador publico usado por milhoes; rejeitado,V6,excluded,84\nx_pous,d_coop,Directed,hospedagem compartilhada com 2.300 dominios; rejeitado,V7,excluded,84\nx_fav,d_real,Directed,clone copiou o favicon da vitima; rejeitado,V9,excluded,84\n"
}

In [ ]:
import urllib.request

RAW_ISCA = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-88/"     # pasta caso-continuo/dados/dia-88 no GitHub

ISCA = {}
for nome, copia in ISCA_EMBUTIDO.items():
    try:                                                     # plano A: versão publicada no repo
        with urllib.request.urlopen(RAW_ISCA + nome, timeout=10) as r:
            ISCA[nome] = r.read().decode("utf-8")
        print(f"{nome}: baixado do GitHub")
    except Exception:                                        # plano B: cópia embutida
        ISCA[nome] = copia
        print(f"{nome}: cópia embutida")

In [ ]:
nos_isca, arestas_isca = ler(ISCA["nos.csv"]), ler(ISCA["arestas.csv"])
gravar("isca/graph", nos_isca, arestas_isca)
print("== check_graph.py no grafo do Isca:"); rodar_check("isca/graph")
print()
ler_grafo(nos_isca, arestas_isca)

rot_i = {n["Id"]: n["Label"] for n in nos_isca}
for vitima, laranja in (("v1", "l01"), ("v3", "l03")):
    for niveis in (("confirmed",), ("confirmed", "likely")):
        t = caminho_por(nos_isca, arestas_isca, ["sms_1", "d_pix", vitima, laranja], niveis)
        print(f"\nSMS -> {rot_i[laranja]} só por {'/'.join(niveis)}:", "nenhum" if t is None else "")
        for e in t or []:
            print(f"   {rot_i[e['Source']]} --[{e['Label']} · {e['confidence']}]--> {rot_i[e['Target']]}")

Repare no **L-03**: só com arestas confirmadas, o caminho chega ao V3 pelo **IP do operador** (que entrou também na conta do V1), não pelo clique do V3. A afirmação "o V3 digitou a senha no site falso" continua *provável* (ele não registrou o clique na ouvidoria). O relatório diz as duas coisas, cada uma com a sua confiança.

In [ ]:
#@title Memorando de leitura do grafo (Isca) { display-mode: "form" }
clusters = "isca (infraestrutura do golpe contra a Serra Azul), outras-coops (Coopvale), vitima (cooperados e cooperativa), laranjas (chaves fictícias L-01..L-03), ponte (ik.js e IP do operador), pergunta-aberta (H1 e H4), excluded"  #@param {type:"string"}
pontes = "art_ik liga isca a outras-coops: 1 tipo de pivô (recurso de página; o comentário 'pescador' está no mesmo arquivo). Firmaria: certificado ou IP de inquilino único compartilhado."  #@param {type:"string"}
hubs = "d_pix (hub real: o domínio da isca); org_sa (artefato do desenho: tudo imita a marca)"  #@param {type:"string"}
excluidos = "lnk.example (encurtador público), pousada-ipe (hospedagem compartilhada), favicon (copiado da vítima)"  #@param {type:"string"}
aresta_mais_consequente = "d_vale -> art_ik: sem ela, some a afirmação 'mesmo kit em outra cooperativa' e muda a recomendação de compartilhar indicadores"  #@param {type:"string"}

memo = (f"# Memorando do grafo — Operação Isca\n\n1. Clusters: {clusters}\n2. Pontes: {pontes}\n3. Hubs: {hubs}\n"
        f"4. Excluídos: {excluidos}\n5. Aresta mais consequente: {aresta_mais_consequente}\n")
Path("notas").mkdir(exist_ok=True)
Path("notas/graph-memo-isca.md").write_text(memo, encoding="utf-8")
Path("isca/graph/case-graph.mmd").write_text(mermaid(nos_isca, arestas_isca) + "\n", encoding="utf-8")
print(memo)
try:
    from google.colab import files
    for a in ("notes/graph/nodes.csv", "notes/graph/edges.csv", "notes/graph/case-graph.mmd",
              "isca/graph/case-graph.mmd", "notas/graph-memo-isca.md"):
        files.download(a)
except ImportError:
    print("(Fora do Colab: arquivos em notes/graph/, isca/graph/ e notas/)")

## ✅ Antes de fechar o Dia 88

- [ ] `check_graph.py` com `0 problems` (capstone e Isca)
- [ ] Caminho confirmado da imitação até o pagamento 000731
- [ ] Pontes com contagem de tipos de pivô; hubs separados entre "importa" e "artefato do desenho"
- [ ] Excluídos visíveis, com motivo
- [ ] Aresta mais consequente nomeada (capstone e Isca)
- [ ] Teste do leitor novo: como o dinheiro saiu, e qual parte é a menos certa

**Amanhã:** Dia 89: o relatório final com confiança calibrada. Para a Orrin Valley, e para a Marta Quintão e a delegacia.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>